# AF2SFS1 — full-mAP intervention and checkpoint-drift audit
Memisahkan efek inference selector dari efek lintasan optimisasi. **Validation-only, tanpa training, tanpa test.**

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import importlib, json, os, shutil, subprocess, sys, tarfile, torch
from pathlib import Path

assert torch.cuda.is_available(), 'Aktifkan T4 GPU.'
REPO = Path('/content/coffee-bean-detection')
BRANCH = 'codex/af2-complementary-mechanisms'
os.chdir('/content')
if REPO.exists():
    shutil.rmtree(REPO)
clone = ['git', 'clone', '--depth', '1', '--branch', BRANCH, 'https://github.com/ediprin/coffee-bean-detection.git', str(REPO)]
for attempt in range(3):
    result = subprocess.run(clone)
    if result.returncode == 0:
        break
    if REPO.exists():
        shutil.rmtree(REPO)
else:
    raise RuntimeError('Git clone gagal tiga kali.')
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'ultralytics==8.4.96', '-e', str(REPO)], check=True)
sys.path.insert(0, str(REPO / 'src'))
importlib.invalidate_caches()
os.chdir(REPO)
print('GPU:', torch.cuda.get_device_name(0))
print('BRANCH:', BRANCH)

In [ ]:
from coffee_detector.drive_project import require_project_artifact, resolve_drive_project_root

required = (
    'bundles/faruq-development-v3-grouped.tar',
    'experiments/faruq-v3-af2-complement-v1/AF2CTRL/AF2CTRL_seed42/weights/best.pt',
    'experiments/faruq-v3-af2-complement-v1/AF2SFS1/AF2SFS1_seed42/weights/best.pt',
    'experiments/faruq-v3-af2-complement-v1/val_reports/AF2CTRL_seed42_result.json',
    'experiments/faruq-v3-af2-complement-v1/val_reports/AF2SFS1_seed42_result.json',
    'experiments/faruq-v3-af2-complement-v1/root_cause/af2sfs1_root_cause.json',
)
PROJECT = resolve_drive_project_root(required_relative_paths=required)
ARCHIVE, CTRL, SFS, CTRL_REPORT, SFS_REPORT, ROOT_REPORT = [require_project_artifact(PROJECT, path) for path in required]
DATA = Path('/content/faruq-development-v3-grouped')
if not (DATA / 'data.yaml').is_file():
    with tarfile.open(ARCHIVE, 'r') as archive:
        archive.extractall('/content', filter='data')
assert (DATA / 'data.yaml').is_file(), DATA
assert not (DATA / 'test').exists(), 'Test tidak boleh tersedia.'
OUTPUT = PROJECT / 'experiments/faruq-v3-af2-complement-v1/root_cause/map_intervention'
OUTPUT.mkdir(parents=True, exist_ok=True)
SUMMARY = OUTPUT / 'af2sfs1_map_intervention.json'
LOG = OUTPUT / 'af2sfs1_map_intervention_run.log'
print('CONTROL:', CTRL)
print('CANDIDATE:', SFS)
print('OUTPUT:', OUTPUT)

In [ ]:
command = [
    sys.executable, '-u', '-m', 'coffee_detector.analysis.af2sfs1_map_intervention',
    '--control-checkpoint', str(CTRL),
    '--candidate-checkpoint', str(SFS),
    '--control-report', str(CTRL_REPORT),
    '--candidate-report', str(SFS_REPORT),
    '--root-cause-report', str(ROOT_REPORT),
    '--data-root', str(DATA),
    '--output-root', str(OUTPUT),
    '--device', '0',
]
print('MENJALANKAN LIMA VALIDATION STATES', flush=True)
with LOG.open('w', encoding='utf-8') as stream:
    process = subprocess.run(command, cwd=REPO, stdout=stream, stderr=subprocess.STDOUT)
print('\n'.join(LOG.read_text(errors='replace').splitlines()[-120:]))
if process.returncode:
    raise RuntimeError(f'Audit gagal: {process.returncode}; log={LOG}')
result = json.loads(SUMMARY.read_text(encoding='utf-8'))
assert result['decision'] == 'INTERPRETABLE'
assert result['training_executed'] is False
assert result['test_images_accessed'] is False
print('SELESAI:', SUMMARY)

In [ ]:
import pandas as pd

display(pd.DataFrame(result['states']).T.style.format('{:.2%}'))
print('DECOMPOSITION')
display(pd.DataFrame(result['decomposition']).T.style.format('{:+.2%}'))
print('FORCED PATH MINUS NORMAL')
display(pd.DataFrame(result['forced_path_minus_normal']).T.style.format('{:+.2%}'))
print('TOP-5 TOTAL IMPROVEMENTS')
display(pd.DataFrame(result['top5_total_improvements']).style.format({key:'{:+.2%}' for key in ('total_gain','direct_selector_effect','optimization_mediated_effect')}))
print('TOP-5 TOTAL REGRESSIONS')
display(pd.DataFrame(result['top5_total_regressions']).style.format({key:'{:+.2%}' for key in ('total_gain','direct_selector_effect','optimization_mediated_effect')}))
print('CHECKPOINT DRIFT')
display(pd.DataFrame(result['checkpoint_drift']['groups']).T)
print('ATTRIBUTION:', result['attribution'])
print('GATES:', result['gates'])
print('TRAINING:', result['training_executed'], '| TEST:', result['test_images_accessed'])
print('SUMMARY:', SUMMARY)